# PerCE Evaluation

PerCE includes built-in time-series-aware evaluation metrics. All metrics use **DTW distance**, which correctly handles temporal structure — unlike tabular evaluation frameworks that use Euclidean distance on flat feature vectors.

This notebook demonstrates:
1. Running `evaluate_batch()` for aggregate results
2. Per-instance metric breakdown
3. Distribution visualisations
4. Understanding the trade-offs between metrics

**No extra packages needed** — everything here is built into PerCE.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from perce import (
    PerCEExplainer,
    evaluate_batch,
    proximity,
    sparsity,
    validity,
    diversity,
)

# Synthetic data — replace with your real dataset
rng = np.random.default_rng(42)
N, C, T = 80, 6, 300

X_train = rng.normal(0, 1, (N, C, T)).astype(np.float32)
y_train = (X_train[:, 0, 50:150].mean(axis=1) > 0).astype(int)

X_test  = rng.normal(0, 1, (20, C, T)).astype(np.float32)
y_test  = (X_test[:, 0, 50:150].mean(axis=1) > 0).astype(int)

def my_model(X_np):
    means = X_np[:, 0, 50:150].mean(axis=1)
    p1 = 1.0 / (1.0 + np.exp(-2 * means))
    return np.stack([1 - p1, p1], axis=1)

print(f"Data: {X_train.shape}, labels: {dict(zip(*np.unique(y_train, return_counts=True)))}")

## 1. Generate Counterfactuals

In [ ]:
explainer = PerCEExplainer(
    model=my_model,
    n_segments=6,
    alpha=0.5,
    beta=0.6,
    k=3,
    random_state=42,
)
explainer.fit(X_train, y_train)

results = explainer.explain_batch(
    X_test,
    target_classes=[1 - int(y) for y in y_test],
    verbose=True,
)

## 2. Aggregate Metrics

In [ ]:
summary = evaluate_batch(results)

print("PerCE Evaluation Results")
print("=" * 45)
print(f"  Instances       : {summary['n_instances']}")
print(f"  Validity rate   : {summary['validity_rate']:.2%}")
print(f"  Proximity       : {summary['proximity_mean']:.4f} ± {summary['proximity_std']:.4f}")
print(f"  Sparsity        : {summary['sparsity_mean']:.4f} ± {summary['sparsity_std']:.4f}")
print(f"  Diversity       : {summary['diversity']:.4f}")
print("=" * 45)
print()
print("Metric guide:")
print("  Validity  → 1.0 = CF always achieves target class")
print("  Proximity ↓ lower = CF stays closer to original")
print("  Sparsity  ↑ higher = fewer segments modified")
print("  Diversity   application-dependent")

## 3. Per-Instance Breakdown

In [ ]:
records = []
for i, r in enumerate(results):
    records.append({
        "instance":       i,
        "valid":          r.is_valid,
        "proximity":      round(r.proximity_score, 4),
        "sparsity":       round(r.sparsity_score, 4),
        "channels_mod":   len(r.channels_modified),
        "segments_mod":   len(r.segments_modified),
        "target_class":   r.target_class,
    })

df = pd.DataFrame(records)
print(df.to_string(index=False))

## 4. Metric Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Proximity
axes[0, 0].hist(df["proximity"], bins=10, color="steelblue", edgecolor="white", alpha=0.85)
axes[0, 0].axvline(df["proximity"].median(), color="tomato", lw=2, linestyle="--",
                   label=f"Median = {df['proximity'].median():.4f}")
axes[0, 0].set_title("Proximity (DTW, normalised) — lower is better", fontweight="bold")
axes[0, 0].set_xlabel("Score"); axes[0, 0].legend()

# Sparsity
axes[0, 1].hist(df["sparsity"], bins=10, color="mediumseagreen", edgecolor="white", alpha=0.85)
axes[0, 1].axvline(df["sparsity"].median(), color="tomato", lw=2, linestyle="--",
                   label=f"Median = {df['sparsity'].median():.4f}")
axes[0, 1].set_title("Sparsity — higher = fewer segments changed", fontweight="bold")
axes[0, 1].set_xlabel("Score"); axes[0, 1].legend()

# Validity
v_counts = df["valid"].value_counts()
axes[1, 0].bar(["Invalid", "Valid"],
               [v_counts.get(False, 0), v_counts.get(True, 0)],
               color=["tomato", "mediumseagreen"], edgecolor="white", width=0.5)
axes[1, 0].set_title(f"Validity — {df['valid'].mean():.0%} valid", fontweight="bold")
axes[1, 0].set_ylabel("Count")

# Channels modified
axes[1, 1].hist(df["channels_mod"], bins=range(C + 2), color="mediumpurple",
                edgecolor="white", alpha=0.85, align="left")
axes[1, 1].set_title("Channels modified per counterfactual", fontweight="bold")
axes[1, 1].set_xlabel("Number of channels")
axes[1, 1].set_xticks(range(C + 1))

plt.suptitle("PerCE Evaluation Summary", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("perce_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Metric Trade-offs

Proximity and sparsity often trade off against validity — making fewer changes (high sparsity, low proximity) can make it harder to flip the class. This plot shows that relationship.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ["mediumseagreen" if v else "tomato" for v in df["valid"]]

axes[0].scatter(df["proximity"], df["sparsity"], c=colors, s=60, alpha=0.8, edgecolors="white")
axes[0].set_xlabel("Proximity (lower = closer to original)")
axes[0].set_ylabel("Sparsity (higher = fewer segments changed)")
axes[0].set_title("Proximity vs Sparsity", fontweight="bold")
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color="mediumseagreen", label="Valid"),
                         Patch(color="tomato",         label="Invalid")])

axes[1].scatter(df["proximity"], df["channels_mod"], c=colors, s=60, alpha=0.8, edgecolors="white")
axes[1].set_xlabel("Proximity")
axes[1].set_ylabel("Channels modified")
axes[1].set_title("Proximity vs Channels Modified", fontweight="bold")
axes[1].legend(handles=[Patch(color="mediumseagreen", label="Valid"),
                         Patch(color="tomato",         label="Invalid")])

plt.tight_layout()
plt.savefig("perce_tradeoffs.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Using Individual Metrics

You can also call each metric function directly on a single pair.

In [ ]:
r = results[0]

prox = proximity(r.original, r.counterfactual)
spar = sparsity( r.original, r.counterfactual, n_segments=6)
val  = validity( r.counterfactual, my_model, r.target_class)

print(f"Instance 0:")
print(f"  Proximity : {prox:.4f}")
print(f"  Sparsity  : {spar:.4f}")
print(f"  Valid     : {val}")

# Diversity across all counterfactuals
div = diversity(np.stack([r.counterfactual for r in results]))
print(f"\nDiversity across all {len(results)} counterfactuals: {div:.4f}")

---

## Metric Reference

| Metric | Formula | Distance | Goal |
|--------|---------|----------|------|
| **Validity** | `f(X') ≠ f(X)` | — | → 1.0 |
| **Proximity** | `DTW(X, X') / (C×T)` | DTW | ↓ lower |
| **Sparsity** | `1 − N_mod_segs / N_segs` | — | ↑ higher |
| **Diversity** | avg pairwise DTW of all CFs | DTW | context-dependent |

All distance-based metrics use DTW with a Sakoe-Chiba band (`dtw_window=0.1` by default), which correctly accounts for temporal shifts unlike Euclidean distance.

**Why not CEval?** The [CEval toolkit](https://pypi.org/project/CEval/) evaluates tabular counterfactuals using Euclidean/Gower distance on flat feature vectors. Time series data requires DTW-based metrics that respect temporal structure — which is why PerCE ships its own evaluation suite.